# 11_hub_production: Standalone Production Benchmark Pipeline

This notebook is fully self-contained and demonstrates the recommended production pipeline for ViT, Global MLP, and Personalized MLP models. It includes Docker setup, model optimization, Triton deployment, and unified benchmarking. No other notebooks are required.

## 1. Docker Setup & Environment Configuration

This notebook uses a dedicated Docker Compose file (`docker-compose-production.yaml`) to launch Triton Inference Server (with GPU support) and a Jupyter environment. All required volumes and environment variables are pre-configured.

**To start the production environment, run:**

```bash
docker compose -f docker-compose-production.yaml up -d
```

**To verify Triton and Jupyter are running:**

```bash
docker ps
docker logs triton_server 2>&1 | tail -20
```

**Model repository:** Place your ONNX models in `./models_triton` before starting.

## 2. Import Required Libraries & Utilities

Import all necessary libraries for model loading, benchmarking, and Triton client interaction. Define helper functions for latency/throughput measurement and result formatting.

In [ ]:
# Import all required libraries
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict

# Triton client
try:
    import tritonclient.http as httpclient
except ImportError:
    httpclient = None
    print("Triton client not installed. Triton benchmarks will be skipped.")

# Helper: Latency/throughput measurement
def benchmark_fn(fn, n_trials=100, sync_cuda=False):
    latencies = []
    for _ in range(n_trials):
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.time()
        fn()
        if sync_cuda and torch.cuda.is_available():
            torch.cuda.synchronize()
        latencies.append(time.time() - start)
    latencies = np.array(latencies)
    return {
        'median_ms': np.percentile(latencies, 50) * 1000,
        'p95_ms': np.percentile(latencies, 95) * 1000,
        'p99_ms': np.percentile(latencies, 99) * 1000,
        'throughput': n_trials / latencies.sum(),
        'latencies': latencies * 1000
    }

# Helper: Print/format results
def print_bench(title, res):
    print(f"{title}")
    print(f"  Median: {res['median_ms']:.2f} ms  P95: {res['p95_ms']:.2f} ms  P99: {res['p99_ms']:.2f} ms  Throughput: {res['throughput']:.1f} samples/sec")


## 3. ViT Benchmark: Compiled Mode, Batch Size 128

Benchmark the ViT encoder in compiled mode at batch size 128, as recommended for production.

In [ ]:
# ViT Benchmark: Compiled Mode, Batch Size 128
import clip
from torchvision import datasets
from torch.utils.data import DataLoader

# Load ViT-L/14 model
vit_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
clip_model, clip_preprocess = clip.load("ViT-L/14", device=vit_device)

# Compile the visual encoder
clip_model.visual = torch.compile(clip_model.visual)

# Prepare dummy data (simulate production batch)
batch_size = 128
# For demo: use random tensor; in prod, use real images preprocessed with clip_preprocess
images = torch.randn(batch_size, 3, 224, 224, device=vit_device)

# Warm-up
with torch.no_grad():
    clip_model.encode_image(images)
    if vit_device.type == "cuda":
        torch.cuda.synchronize()

# Benchmark
def vit_infer():
    with torch.no_grad():
        clip_model.encode_image(images)
        if vit_device.type == "cuda":
            torch.cuda.synchronize()

vit_res = benchmark_fn(vit_infer, n_trials=50, sync_cuda=(vit_device.type=="cuda"))
print_bench("ViT Compiled, Batch=128", vit_res)


## 4. Global MLP Benchmark: Dynamic Quantized, Optimal Batch Size

Benchmark the Global MLP using dynamic quantization at the optimal batch size for production.

In [ ]:
# Global MLP Benchmark: Dynamic Quantized, Optimal Batch Size
class GlobalMLP(nn.Module):
    def __init__(self, input_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return torch.sigmoid(self.net(x))

# Load model weights (replace with actual path if needed)
global_mlp = GlobalMLP()
global_mlp.load_state_dict(torch.load("workspace/models/inference_only/flickr_global_best_inference_only.pth", map_location="cpu"))
global_mlp.eval()

# Apply dynamic quantization
quantized_mlp = torch.quantization.quantize_dynamic(global_mlp, {nn.Linear}, dtype=torch.qint8)

# Optimal batch size (from benchmarks): 32
mlp_batch = 32
mlp_input = torch.randn(mlp_batch, 768)

# Warm-up
with torch.no_grad():
    quantized_mlp(mlp_input)

# Benchmark
def mlp_infer():
    with torch.no_grad():
        quantized_mlp(mlp_input)

mlp_res = benchmark_fn(mlp_infer, n_trials=100)
print_bench("Global MLP (Dynamic Quantized, Batch=32)", mlp_res)


## 5. Personalized MLP Benchmark: Graph Optimized, Optimal Batch Size

Benchmark the Personalized MLP using graph optimization at the optimal batch size for production.

In [ ]:
# Personalized MLP Benchmark: Graph Optimized, Optimal Batch Size
class PersonalizedMLP(nn.Module):
    def __init__(self, num_users, input_dim=768, user_dim=16):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, user_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim + user_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x, user_idx):
        u = self.user_embedding(user_idx)
        z = torch.cat([x, u], dim=-1)
        return torch.sigmoid(self.net(z))

# Load model weights (replace with actual path if needed)
p_state = torch.load("workspace/models/inference_only/flickr_personalized_best_inference_only.pth", map_location="cpu")
num_users = p_state["user_embedding.weight"].shape[0]
user_dim = p_state["user_embedding.weight"].shape[1]
personal_mlp = PersonalizedMLP(num_users=num_users, user_dim=user_dim)
personal_mlp.load_state_dict(p_state)
personal_mlp.eval()

# Graph optimization (TorchScript)
personal_mlp_script = torch.jit.script(personal_mlp)

# Optimal batch size: 256
mlp_batch = 256
mlp_input = torch.randn(mlp_batch, 768)
user_idx = torch.zeros(mlp_batch, dtype=torch.long)

# Warm-up
with torch.no_grad():
    personal_mlp_script(mlp_input, user_idx)

# Benchmark
def personal_infer():
    with torch.no_grad():
        personal_mlp_script(mlp_input, user_idx)

personal_res = benchmark_fn(personal_infer, n_trials=100)
print_bench("Personalized MLP (Graph Optimized, Batch=256)", personal_res)


## 6. Global MLP: Triton Inference Server (2 Instances + Dynamic Batching)

Deploy and benchmark the Global MLP on Triton with 2 instances and dynamic batching (preferred_batch_size: [4,8,16,32,64,128]).

In [ ]:
# Triton Benchmark: Global MLP (2 Instances + Dynamic Batching)
if httpclient is not None:
    TRITON_URL = "localhost:8000"
    client = httpclient.InferenceServerClient(url=TRITON_URL)
    
    # Prepare input (batch=32)
    batch_emb = np.random.randn(32, 768).astype(np.float32)
    def infer_global_triton():
        inputs = [httpclient.InferInput("input", batch_emb.shape, "FP32")]
        inputs[0].set_data_from_numpy(batch_emb)
        outputs = [httpclient.InferRequestedOutput("output")]
        result = client.infer(model_name="flickr_global", inputs=inputs, outputs=outputs)
        return result.as_numpy("output")
    
    triton_res = benchmark_fn(infer_global_triton, n_trials=100)
    print_bench("Triton Global MLP (2 inst, dyn batch, b=32)", triton_res)
else:
    print("Triton client not available. Skipping Triton Global MLP benchmark.")


## 7. Personalized MLP: Triton Inference Server (2 Instances + Dynamic Batching)

Deploy and benchmark the Personalized MLP on Triton with 2 instances and dynamic batching (preferred_batch_size: [4,8,16,32,64,128]).

In [ ]:
# Triton Benchmark: Personalized MLP (2 Instances + Dynamic Batching)
if httpclient is not None:
    # Prepare input (batch=256)
    batch_emb = np.random.randn(256, 768).astype(np.float32)
    user_idx = np.zeros((256, 1), dtype=np.int64)
    def infer_personal_triton():
        inp_emb = httpclient.InferInput("embedding", batch_emb.shape, "FP32")
        inp_emb.set_data_from_numpy(batch_emb)
        inp_idx = httpclient.InferInput("user_idx", user_idx.shape, "INT64")
        inp_idx.set_data_from_numpy(user_idx)
        outputs = [httpclient.InferRequestedOutput("output")]
        result = client.infer(model_name="flickr_personalized", inputs=[inp_emb, inp_idx], outputs=outputs)
        return result.as_numpy("output")
    triton_p_res = benchmark_fn(infer_personal_triton, n_trials=100)
    print_bench("Triton Personalized MLP (2 inst, dyn batch, b=256)", triton_p_res)
else:
    print("Triton client not available. Skipping Triton Personalized MLP benchmark.")


## 8. Unified Production Results Summary

Aggregate all benchmark results, visualize with bar charts and latency plots, and provide a final production recommendation table.

In [ ]:
# Aggregate and visualize results
results = []
if 'vit_res' in globals():
    results.append({
        'Model': 'ViT (Compiled)', 'Mode': 'Standalone', 'Batch': 128,
        'Median (ms)': vit_res['median_ms'], 'P95 (ms)': vit_res['p95_ms'], 'Throughput': vit_res['throughput']
    })
if 'mlp_res' in globals():
    results.append({
        'Model': 'Global MLP (Dyn Quant)', 'Mode': 'Standalone', 'Batch': 32,
        'Median (ms)': mlp_res['median_ms'], 'P95 (ms)': mlp_res['p95_ms'], 'Throughput': mlp_res['throughput']
    })
if 'personal_res' in globals():
    results.append({
        'Model': 'Personalized MLP (Graph Opt)', 'Mode': 'Standalone', 'Batch': 256,
        'Median (ms)': personal_res['median_ms'], 'P95 (ms)': personal_res['p95_ms'], 'Throughput': personal_res['throughput']
    })
if 'triton_res' in globals():
    results.append({
        'Model': 'Global MLP', 'Mode': 'Triton', 'Batch': 32,
        'Median (ms)': triton_res['median_ms'], 'P95 (ms)': triton_res['p95_ms'], 'Throughput': triton_res['throughput']
    })
if 'triton_p_res' in globals():
    results.append({
        'Model': 'Personalized MLP', 'Mode': 'Triton', 'Batch': 256,
        'Median (ms)': triton_p_res['median_ms'], 'P95 (ms)': triton_p_res['p95_ms'], 'Throughput': triton_p_res['throughput']
    })

if results:
    df = pd.DataFrame(results)
    display(df)
    df.plot(x='Model', y='Throughput', kind='bar', legend=False, title='Throughput (samples/sec)')
    plt.ylabel('Throughput (samples/sec)')
    plt.show()
    df.plot(x='Model', y=['Median (ms)', 'P95 (ms)'], kind='bar', title='Latency (ms)')
    plt.ylabel('Latency (ms)')
    plt.show()
else:
    print("No benchmark results to summarize.")

# Final production recommendation table
from IPython.display import display, Markdown
rec_table = """
| Model | Optimization | Batch | Serving | Triton Settings | Rec. Use |
|-------|--------------|-------|---------|-----------------|----------|
| ViT   | Compiled     | 128   | Standalone | N/A           | Feature extraction |
| Global MLP | Dyn Quant | 32 | Triton | 2 inst, dyn batching [4,8,16,32,64,128] | Scoring |
| Personalized MLP | Graph Opt | 256 | Triton | 2 inst, dyn batching [4,8,16,32,64,128] | Scoring |
"""
display(Markdown(rec_table))
